# Project 2 Template — Build and Test a Grounded Assistant (RAG)
**AI Models and Agents for Business · Project 2: Build a Grounded Assistant**

This notebook builds **Version 2** and **Version 3** of your assistant (Version 1 is your no-code Gemini Gem). It answers questions from *your* documents, shows you exactly which passages it used, and helps you score it on your 20-question test set.

### How the assistant works (RAG in four steps)
1. **Chunk** — cut your documents into overlapping pieces of N words.
2. **Retrieve** — for each question, find the K pieces most similar to it.
3. **Prompt** — hand those pieces to the model with an instruction ("answer only from these").
4. **Answer** — the model writes a reply. You judge it.

You edit three things: **chunk size, number of passages (K), and the instruction text.** The code is provided. Embedding and search run free on the Colab CPU with no key; only the final answer step uses Groq.

> **Write your test set before you build anything: 20 development questions and 8 held-out questions.** The development questions are for building: look at them, tune against them, fix what fails. The 8 held-out questions stay sealed until the very end. You run your final version on them **once**, and that is the honest estimate of how the assistant will do on questions nobody tuned it for.

### One-time setup: store your key in Colab Secrets
1. Click the **🔑 key icon** in the left sidebar of Colab.
2. Click **Add new secret**. Name: `GROQ_API_KEY`. Value: your key (it starts with `gsk_`).
3. Turn on **Notebook access** for this notebook.

Colab will remember the key for all your notebooks, so you never paste it into a cell. If you skip this, the setup cell will simply ask you to paste the key each time (the box hides what you type).

In [ ]:
# Run this cell first. Do not modify.
!pip install groq --quiet

import time, json, textwrap
from groq import Groq
import groq

def _get_key():
    """Look for the key in Colab Secrets first; otherwise ask for it (hidden as you type)."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

API_KEY = _get_key()
client = Groq(api_key=API_KEY)
MODEL = "openai/gpt-oss-20b"         # if Groq retires this name, change it here only

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

def _retry_after(err, default=15):
    """Seconds the free tier asks us to wait (from the error's retry-after header), capped at 60."""
    try:
        return min(60.0, float(err.response.headers.get("retry-after", default)))
    except Exception:
        return float(default)

def ask(prompt, system=None, temperature=0.0, max_tokens=300, messages=None, model=None):
    """Send one request to the model and return the full response object.
    Waits and retries if the free tier says 'slow down'."""
    if messages is None:
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    for attempt in range(6):
        try:
            return client.chat.completions.create(model=model or MODEL, messages=messages,
                                                  temperature=temperature, max_tokens=max_tokens)
        except groq.RateLimitError as err:
            wait = _retry_after(err)
            print(f"(rate limit hit - waiting {wait:.0f} seconds, then trying again)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited. Wait a minute and re-run this cell.")

def say(prompt, **kw):
    """Same as ask(), but returns just the text of the answer."""
    return (ask(prompt, **kw).choices[0].message.content or "").strip()

print("✅ Setup complete. Model:", MODEL)

# Retrieval libraries (scikit-learn ships with Colab; sentence-transformers usually does too)
!pip install sentence-transformers --quiet

## Part 1 — ✏️ Configure
Run it as is first: it uses a tiny **sample company** and a sample test (6 development questions and 2 held-out) so that you can see the whole pipeline work in a couple of minutes. Then set `USE_SAMPLE = False`, upload your corpus files and your test set, and make the settings your own.

**Your test set CSV needs these columns** (the example in the Corpora folder has the first five): `q_id`, `question_type` (`direct`, `two_passage` or `unanswerable`), `question`, `correct_answer`, `source_sections` (like `4.2` or `5.3; 12.3`, or `none`), and `split` (`dev` for the 20 development questions, `holdout` for the 8 held-out ones). If there is no `split` column, every question is treated as `dev`.

In [ ]:
# ✏️ CHANGE ME ---------------------------------------------------------
USE_SAMPLE   = True      # True = built-in sample company. False = upload your own files in Part 2.

CHUNK_WORDS  = 150       # how many words per chunk
OVERLAP      = 30        # words shared between neighbouring chunks
TOP_K        = 3         # how many passages to give the model
RETRIEVER    = "embeddings"   # "embeddings" (meaning-based) or "tfidf" (keyword-based, always works)

PAUSE_SECONDS = 2        # raise this if you see rate-limit messages

# Version 2: the plain, short instruction
V2_INSTRUCTION = "Answer the question using the passages provided."

# Version 3: your instruction file. This is the same idea as an AGENTS.md: standing rules the assistant must follow.
# Write rules a new employee could follow. Test each rule on the questions it is meant to fix.
V3_INSTRUCTION = """You are the help assistant for the organization whose documents are provided in the passages.

RULES
1. Answer ONLY from the passages. Do not use outside knowledge, even if you are sure.
2. After every factual claim, cite the section number in brackets, like [4.2]. Use only section numbers that appear in the passages.
3. If the passages do not contain the answer, reply exactly: "I can't find that in the documents." Then suggest who the person could ask. Do not guess.
4. If the question assumes something the passages do not say, point that out instead of accepting it.
5. If the question needs two sections, use both and cite both.
6. Do not give legal, medical or tax advice. Say a person in the organization should answer.
7. Keep answers to three sentences or fewer.
"""
# ----------------------------------------------------------------------
print("Settings loaded. Chunk =", CHUNK_WORDS, "words, overlap =", OVERLAP, ", top-k =", TOP_K, ", retriever =", RETRIEVER)

## Part 2 — Load the documents and the test set

In [ ]:
import pandas as pd, io, re, math, json, time, random
import numpy as np

SAMPLE_DOC = """# Maple Street Bike Co. Customer Handbook
*Maple Street Bike Co. is a fictional company used for this demo.*

## 1. Service
### 1.1 Tune-ups
A basic tune-up costs $65 and takes two business days. It includes brake and gear adjustment, a chain clean and a tire check. Appointments can be booked online or in the store.
### 1.2 Repairs
Repairs are quoted before work begins. If you approve the quote, the shop starts within three business days. Parts are guaranteed for 90 days.

## 2. Returns
### 2.1 Bikes
New bikes can be returned within 14 days if they have not been ridden outdoors. A 10 percent restocking fee applies to every returned bike.
### 2.2 Accessories
Accessories such as helmets and locks can be returned within 30 days with a receipt. Helmets that have been worn outdoors cannot be returned.

## 3. Membership
### 3.1 Ride Club
Ride Club costs $40 per year. Members receive 10 percent off accessories and one free tune-up each year.
### 3.2 Cancellation
Ride Club can be cancelled at any time. The fee is not refunded, but the free tune-up can still be used until the membership year ends.

## 4. Warranty
### 4.1 Frames
Every new bike frame carries a five-year warranty against manufacturing defects. The warranty covers cracks and failed welds. It does not cover damage from crashes, from riding in competitions, or from carrying loads above the weight limit printed on the frame.
### 4.2 Components
Components such as brakes, gears, wheels and saddles are covered for one year. Normal wear items such as tires, brake pads, chains and cables are not covered. To make a claim, bring the bike and the original receipt to any Maple Street store.

## 5. Stores and Hours
### 5.1 Opening Hours
The downtown store is open Monday to Saturday from 9 a.m. to 6 p.m. and on Sunday from 11 a.m. to 4 p.m. The riverside store is open every day from 10 a.m. to 7 p.m. in summer. In winter, from November through March, the riverside store closes at 5 p.m.
### 5.2 Holidays
Both stores close on New Year's Day, Thanksgiving Day and Christmas Day. On the day before each of these holidays the stores close two hours early.

## 6. Delivery and Assembly
### 6.1 Delivery
Bikes bought online are delivered within five business days to addresses within 40 miles of either store. Delivery costs $30 for bikes and is free for orders over $1,500. Accessories ship free on orders over $75.
### 6.2 Assembly
Bikes are delivered fully assembled and tuned. If you choose in-store pickup, the bike is also assembled and given a safety check at no charge. Pedals and handlebars on boxed bikes shipped outside the delivery area must be fitted by the customer.
"""

SAMPLE_TESTS = pd.DataFrame([
 {"q_id": "Q1", "question_type": "direct", "question": "How much does a basic tune-up cost?", "correct_answer": "$65.", "source_sections": "1.1"},
 {"q_id": "Q2", "question_type": "direct", "question": "How long do I have to return a new bike?", "correct_answer": "14 days, if it has not been ridden outdoors.", "source_sections": "2.1"},
 {"q_id": "Q3", "question_type": "direct", "question": "What do Ride Club members get?", "correct_answer": "10 percent off accessories and one free tune-up per year.", "source_sections": "3.1"},
 {"q_id": "Q4", "question_type": "two_passage", "question": "If I cancel Ride Club, can I still use my free tune-up?", "correct_answer": "Yes, until the membership year ends; the fee is not refunded.", "source_sections": "3.1; 3.2"},
 {"q_id": "Q5", "question_type": "unanswerable", "question": "Do you sell electric bikes?", "correct_answer": "The handbook does not say.", "source_sections": "none"},
 {"q_id": "Q6", "question_type": "unanswerable", "question": "Can I rent a bike for a weekend?", "correct_answer": "The handbook does not say.", "source_sections": "none"},
 # Held-out questions: sealed until the final test in Part 9
 {"q_id": "H1", "question_type": "direct", "question": "Are the stores open on Thanksgiving?", "correct_answer": "No, both stores close on Thanksgiving Day.", "source_sections": "5.2", "split": "holdout"},
 {"q_id": "H2", "question_type": "unanswerable", "question": "Do you offer professional bike fittings?", "correct_answer": "The handbook does not say.", "source_sections": "none", "split": "holdout"},
])
SAMPLE_TESTS["split"] = SAMPLE_TESTS["split"].fillna("dev")

if USE_SAMPLE:
    docs = {"maple_street_handbook.md": SAMPLE_DOC}
    tests = SAMPLE_TESTS.copy()
    print("Using the built-in sample company and 8 sample questions (6 development, 2 held-out).")
else:
    from google.colab import files
    print("STEP 1: choose your document file(s) (.md or .txt). You can select several.")
    up = files.upload()
    docs = {name: data.decode("utf-8", errors="ignore") for name, data in up.items()}
    print("STEP 2: choose your TEST SET csv.")
    up2 = files.upload()
    tests = pd.read_csv(io.BytesIO(up2[list(up2.keys())[0]]))
    print("Loaded", len(docs), "document(s) and", len(tests), "test questions.")

for name, text in docs.items():
    print(f"{name}: {len(text.split()):,} words")

# Split the test set. `tests` = development questions (used in Parts 4-8).
# `holdout` = sealed questions, used ONCE in Part 9. Don't read their answers while you build.
if "split" not in tests.columns:
    tests["split"] = "dev"
tests["split"] = tests["split"].fillna("dev").astype(str).str.strip().str.lower()
all_tests = tests.copy()
holdout = all_tests[all_tests["split"] == "holdout"].reset_index(drop=True)
tests = all_tests[all_tests["split"] != "holdout"].reset_index(drop=True)
print(f"Development questions: {len(tests)}   Held-out questions (sealed until Part 9): {len(holdout)}")
print(tests["question_type"].value_counts().to_string())

## Part 3 — Chunk and index
Documents are turned into a stream of words. Each chunk is `CHUNK_WORDS` long and remembers **which section numbers it touches**, so that you can later check whether retrieval found the right place.

*Why overlap?* A fact on the boundary between two chunks would otherwise be cut in half.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def section_id(heading):
    m = re.match(r"^\s*(\d+(?:\.\d+)*)\b", heading)
    return m.group(1) if m else None

def make_chunks(docs, chunk_words, overlap):
    chunks = []
    for name, text in docs.items():
        numbered = any(re.match(r"^#+\s*\d", ln) for ln in text.split("\n"))
        words, secs, cur = [], [], "front"
        for line in text.split("\n"):
            if line.startswith("#"):
                sid = section_id(line.lstrip("#"))
                if sid: cur = sid
                elif not numbered: cur = line.lstrip("#").strip()[:40]     # unnumbered documents: use the heading text
            for w in line.split():
                words.append(w); secs.append(cur)
        step = max(1, chunk_words - overlap)
        for start in range(0, max(1, len(words)), step):
            piece = words[start:start+chunk_words]
            if not piece: break
            chunks.append({"doc": name, "text": " ".join(piece),
                           "sections": list(dict.fromkeys(secs[start:start+chunk_words])),
                           "start": start})
            if start + chunk_words >= len(words): break
    return chunks

_st_model = None
def get_embedder():
    global _st_model
    if _st_model is None:
        from sentence_transformers import SentenceTransformer
        _st_model = SentenceTransformer("all-MiniLM-L6-v2")
    return _st_model

class Index:
    def __init__(self, chunks, retriever):
        self.chunks = chunks
        self.retriever = retriever
        texts = [c["text"] for c in chunks]
        if retriever == "embeddings":
            try:
                self.model = get_embedder()
                self.mat = self.model.encode(texts, normalize_embeddings=True)
            except Exception as e:
                print("(embedding model unavailable:", type(e).__name__, "- using keyword search instead)")
                self.retriever = "tfidf"
        if self.retriever == "tfidf":
            self.vec = TfidfVectorizer(stop_words="english")
            self.mat = self.vec.fit_transform(texts)
    def search(self, query, k):
        if self.retriever == "embeddings":
            q = self.model.encode([query], normalize_embeddings=True)[0]
            sims = self.mat @ q
        else:
            q = self.vec.transform([query])
            sims = (self.mat @ q.T).toarray().ravel()
        top = np.argsort(-sims)[:k]
        return [(float(sims[i]), self.chunks[i]) for i in top]

t0 = time.time()
chunks = make_chunks(docs, CHUNK_WORDS, OVERLAP)
index = Index(chunks, RETRIEVER)
print(f"{len(chunks)} chunks built; retriever = {index.retriever}; took {time.time()-t0:.1f} s")
print("\nExample chunk:\n", chunks[0]["sections"], "->", chunks[0]["text"][:300], "...")

## Part 4 — Look before you score: what does retrieval find?
Pick any question. This shows the passages the assistant would receive. **Most wrong answers are retrieval failures, not model failures** — the right passage never reached the model. Read these before you blame the model.

In [ ]:
# ✏️ CHANGE ME — try any question, including one the documents cannot answer
QUESTION = tests.iloc[0]["question"]

for score, c in index.search(QUESTION, TOP_K):
    print(f"score {score:.2f} | sections {c['sections']} | {c['doc']}")
    print("   ", c["text"][:250], "...\n")

## Part 5 — Ask the assistant (one question)
`answer()` builds the prompt from the retrieved passages and your instruction, and asks the model. Try your own instruction text and watch what changes.

In [ ]:
def build_prompt(question, retrieved):
    parts = []
    for i, (score, c) in enumerate(retrieved, 1):
        parts.append(f"[Passage {i} | Sections: {', '.join(c['sections'])}]\n{c['text']}")
    return "PASSAGES\n" + "\n\n".join(parts) + f"\n\nQUESTION: {question}"

def answer(question, instruction, k=None, idx=None):
    idx = idx or index
    retrieved = idx.search(question, k or TOP_K)
    resp = ask(build_prompt(question, retrieved), system=instruction, temperature=0.0, max_tokens=300)
    secs = sorted({s for _, c in retrieved for s in c["sections"]})
    return resp.choices[0].message.content.strip(), secs, resp.usage

Q = tests.iloc[0]["question"]
for label, ins in [("V2 (plain instruction)", V2_INSTRUCTION), ("V3 (your instruction file)", V3_INSTRUCTION)]:
    text, secs, usage = answer(Q, ins)
    print(f"--- {label}\nQ: {Q}\nA: {text}\n(passages came from sections {secs}; {usage.prompt_tokens} tokens in)\n")

## Part 6 — Run the whole test set and save the scoring sheet
This runs every **development** question through **Version 2** and **Version 3** and saves two CSV files. The notebook adds one automatic column, `retrieval_hit` (did retrieval bring back the section(s) the answer lives in?). **You** fill in the rest by hand in Google Sheets:

| column | what to enter |
|---|---|
| `correct` | `1` if the answer matches `correct_answer` in substance; for `unanswerable` questions `1` means it **declined** instead of inventing an answer |
| `cited` | `1` if it cited a section that really supports the claim (answerable questions only) |
| `error_type` | for each `0`: a short label, e.g. `made up number`, `wrong section`, `missed second passage`, `refused but answer was there` |
| `notes` | judgment calls |

Expect 1–2 minutes per version for 20 questions.

In [ ]:
def expected_sections(s):
    s = str(s).strip().lower()
    return [] if s in ("none", "nan", "") else [x.strip() for x in s.split(";") if x.strip()]

def run_version(label, instruction, k=None, idx=None, questions=None):
    questions = tests if questions is None else questions
    rows = []
    for i, r in questions.iterrows():
        text, secs, usage = answer(r["question"], instruction, k, idx)
        exp = expected_sections(r["source_sections"])
        hit = (all(e in secs for e in exp) if exp else None)       # None = nothing to retrieve (unanswerable)
        rows.append({"q_id": r["q_id"], "question_type": r["question_type"], "question": r["question"],
                     "correct_answer": r["correct_answer"], "assistant_answer": text.replace("\n", " "),
                     "expected_sections": r["source_sections"], "retrieved_sections": "; ".join(secs),
                     "retrieval_hit": hit, "tokens_in": usage.prompt_tokens, "tokens_out": usage.completion_tokens,
                     "correct": "", "cited": "", "error_type": "", "notes": ""})
        print(f"[{label}] {i+1}/{len(questions)} {r['q_id']} ({r['question_type']}) retrieval_hit={hit}")
        time.sleep(PAUSE_SECONDS if PAUSE_SECONDS else 0)
    return pd.DataFrame(rows).astype({"correct": "object", "cited": "object", "error_type": "object", "notes": "object"})

res2 = run_version("V2", V2_INSTRUCTION)
res3 = run_version("V3", V3_INSTRUCTION)
res2.to_csv("results_v2.csv", index=False)
res3.to_csv("results_v3.csv", index=False)
open("AGENTS.md", "w").write("# Instruction file (Version 3)\n\n" + V3_INSTRUCTION)
print("\nSaved results_v2.csv, results_v3.csv, AGENTS.md")
try:
    from google.colab import files
    for f in ["results_v2.csv", "results_v3.csv", "AGENTS.md"]: files.download(f)
except Exception as e:
    print("(download not available here:", type(e).__name__, ")")

## Part 7 — Score report
Upload your **hand-labeled** `results_v2.csv` and `results_v3.csv`. Until you have, `UPLOAD_LABELED = False` shows a **demo with made-up labels**. The demo numbers mean nothing.

The report shows, for each version: correct rate by question type, citation rate, and retrieval hit rate. It also splits every wrong answer into two kinds: **retrieval problem** (the right section never reached the model) and **generation problem** (the model had it and still got it wrong). They have different fixes.

In [ ]:
UPLOAD_LABELED = False       # ✏️ set to True after labeling

def load_labeled(default_df, name, upload=None):
    upload = UPLOAD_LABELED if upload is None else upload
    if upload:
        from google.colab import files
        print(f"Upload your labeled {name}...")
        up = files.upload()
        return pd.read_csv(io.BytesIO(up[list(up.keys())[0]])), False
    demo = default_df.copy()
    rng = random.Random(1)
    demo["correct"] = [rng.choice([1, 1, 0]) for _ in range(len(demo))]
    demo["cited"] = [rng.choice([1, 0]) if t != "unanswerable" else "" for t in demo["question_type"]]
    demo["error_type"] = ["" if c == 1 else "demo error" for c in demo["correct"]]
    return demo, True

def report(df, label, demo):
    d = df[df["correct"].astype(str).str.strip().isin(["0", "1", "0.0", "1.0"])].copy()
    if len(d) == 0:
        raise ValueError(f"No rows in {label} have correct = 1 or 0. Fill in the correct column first.")
    d["correct"] = d["correct"].astype(float).astype(int)
    print(("*** DEMO LABELS — NOT REAL RESULTS ***\n" if demo else "") + f"=== {label}: {int(d['correct'].sum())}/{len(d)} correct ({d['correct'].mean():.0%})")
    print(d.groupby("question_type")["correct"].agg(["sum", "count", "mean"]).rename(columns={"sum": "correct", "count": "n", "mean": "rate"}).round(2).to_string())
    ans = d[d["question_type"] != "unanswerable"].copy()
    cited = pd.to_numeric(ans["cited"], errors="coerce").dropna()
    if len(cited): print(f"Cited a supporting section (answerable questions): {cited.mean():.0%}")
    hits = ans["retrieval_hit"].astype(str).str.lower().map({"true": 1, "false": 0}).dropna()
    if len(hits): print(f"Retrieval found the right section(s): {hits.mean():.0%} of answerable questions")
    wrong = ans[ans["correct"] == 0].copy()
    if len(wrong):
        wrong["kind"] = wrong["retrieval_hit"].astype(str).str.lower().map({"true": "generation problem", "false": "retrieval problem"})
        print("Wrong answers, by cause:", wrong["kind"].value_counts().to_dict())
    un = d[d["question_type"] == "unanswerable"]
    if len(un): print(f"Appropriate refusals (unanswerable): {int(un['correct'].sum())}/{len(un)}")
    ti, to = d["tokens_in"].mean(), d["tokens_out"].mean()
    print(f"Average tokens per question: {ti:.0f} in, {to:.0f} out")
    print()
    return d

lab2, demo2 = load_labeled(res2, "results_v2.csv")
lab3, demo3 = load_labeled(res3, "results_v3.csv")
s2 = report(lab2, "Version 2 (plain instruction)", demo2)
s3 = report(lab3, "Version 3 (your instruction file)", demo3)

## Part 8 — Retrieval experiment (no model calls, no cost)
How do chunk size and the number of passages change whether retrieval finds the right place? This sweep needs **no API calls**, so it runs fast. It uses the `source_sections` column of your test set, so it only measures answerable questions.

**Write your prediction first:** do bigger chunks help or hurt? Does K = 5 beat K = 3?

In [ ]:
# ✏️ CHANGE ME — (chunk size, overlap, top-k) combinations to try
SWEEP = [(60, 10, 3), (100, 20, 3), (150, 30, 3), (250, 50, 3), (400, 80, 3), (150, 30, 1), (150, 30, 5)]
PREDICTION = "We predict that ____ because ____."

answerable = tests[tests["question_type"] != "unanswerable"]
rows = []
for cw, ov, k in SWEEP:
    ch = make_chunks(docs, cw, ov)
    ix = Index(ch, RETRIEVER)
    hits = []
    for _, r in answerable.iterrows():
        secs = {s for _, c in ix.search(r["question"], k) for s in c["sections"]}
        exp = expected_sections(r["source_sections"])
        hits.append(all(e in secs for e in exp))
    approx_tokens = int(k * cw * 1.3)
    rows.append({"chunk_words": cw, "overlap": ov, "top_k": k, "chunks": len(ch),
                 "retrieval_hit_rate": round(sum(hits)/len(hits), 2), "~prompt_tokens_for_passages": approx_tokens})
sweep_df = pd.DataFrame(rows)
print(sweep_df.to_string(index=False))
print("\nA higher hit rate is better, but notice the last column: more text in the prompt costs more and can distract the model.")
sweep_df.to_csv("retrieval_sweep.csv", index=False)

## Part 9 — Final test on the held-out questions (run this once)
Up to now you have been tuning against your 20 development questions: you read the failures, changed chunk size, K and the instruction, and re-ran. That is how you build, but it means your development score is **optimistic**. Your changes were chosen because they fixed *those* questions.

When your design is final, run it **once** on the 8 held-out questions and report that score next to the development score. Do not go back and tune after you see the held-out results; if you do, they stop being held out. If the held-out score is much worse, say so in your memo: that gap is a finding about how well your assistant will do on questions nobody tuned it for.

1. Set `FINAL_INSTRUCTION` to the version you will recommend (V2 or V3), and make sure the settings in Part 1 are your final ones.
2. Run the cell. It saves `results_final_holdout.csv`. Label it the same way as before.
3. Set `UPLOAD_HOLDOUT_LABELED = True`, re-run, and upload the labeled file.

In [ ]:
# ✏️ CHANGE ME — your final design
FINAL_LABEL = "V3"                 # "V2" or "V3": the version you will recommend
FINAL_INSTRUCTION = V3_INSTRUCTION if FINAL_LABEL == "V3" else V2_INSTRUCTION
UPLOAD_HOLDOUT_LABELED = False     # ✏️ set to True after labeling results_final_holdout.csv

if len(holdout) == 0:
    print("No held-out questions found. Add a 'split' column to your test set with 'holdout' on 8 questions.")
else:
    res_final = run_version("FINAL", FINAL_INSTRUCTION, questions=holdout)
    res_final.to_csv("results_final_holdout.csv", index=False)
    try:
        from google.colab import files
        if not UPLOAD_HOLDOUT_LABELED: files.download("results_final_holdout.csv")
    except Exception as e:
        print("(download not available here:", type(e).__name__, ")")

    lab_final, demo_final = load_labeled(res_final, "results_final_holdout.csv", upload=UPLOAD_HOLDOUT_LABELED)
    s_final = report(lab_final, f"{FINAL_LABEL} on the HELD-OUT questions", demo_final)
    s_dev = s3 if FINAL_LABEL == "V3" else s2
    print(f"Development score : {s_dev['correct'].mean():.0%} of {len(s_dev)} questions")
    print(f"Held-out score    : {s_final['correct'].mean():.0%} of {len(s_final)} questions")
    print(f"With {len(s_final)} held-out questions, one answer moves the score by about {100 / len(s_final):.0f} points,")
    print("so compare the two with care. A big drop means your design was tuned to the development questions.")


## Part 10 — Version comparison table for your memo
Type your **Version 1 (Gemini Gem)** results below. The table then lines up all three versions. Fill in the build time honestly (hours your team spent on each version) — the memo asks which one a firm should deploy.

In [ ]:
# ✏️ CHANGE ME — Version 1 numbers from your Gem scoring, and honest build times
V1 = {"correct": 12, "of": 20, "cited": "n/a", "refusals": "2/5", "build_hours": 0.5}
BUILD_HOURS = {"V2": 3.0, "V3": 1.0}      # V3 time is the extra time to write and test the instruction file

def row(label, d, hours):
    d = d.copy()
    un = d[d["question_type"] == "unanswerable"]
    ans = d[d["question_type"] != "unanswerable"]
    cited = pd.to_numeric(ans["cited"], errors="coerce").dropna()
    per_q = (d["tokens_in"].mean()/1e6*PRICE_IN + d["tokens_out"].mean()/1e6*PRICE_OUT)
    return {"Version": label, "Correct": f"{int(d['correct'].sum())}/{len(d)}",
            "Cited": f"{cited.mean():.0%}" if len(cited) else "n/a",
            "Refusals": f"{int(un['correct'].sum())}/{len(un)}" if len(un) else "n/a",
            "Build hours": hours, "Model $ per 1,000 questions": round(per_q*1000, 4)}

PRICE_IN, PRICE_OUT = 0.075, 0.30      # ✏️ dollars per million tokens (gpt-oss-20b list price, Oct 2026: check Groq's current prices)

table = pd.DataFrame([
    {"Version": "V1 Gemini Gem", "Correct": f"{V1['correct']}/{V1['of']}", "Cited": V1["cited"], "Refusals": V1["refusals"],
     "Build hours": V1["build_hours"], "Model $ per 1,000 questions": "n/a"},
    row("V2 RAG, plain instruction", s2, BUILD_HOURS["V2"]),
    row("V3 RAG + instruction file", s3, BUILD_HOURS["V3"]),
])
print(("*** CONTAINS DEMO LABELS ***\n" if (demo2 or demo3) else "") + table.to_string(index=False))
print("\nWhich version would you deploy, and what would change your mind? (That is your memo.)")

---
## ✅ Hand-in checklist
* This notebook, run top to bottom, with your settings.
* `results_v2.csv` and `results_v3.csv` with your hand-labels, and `AGENTS.md`.
* `results_final_holdout.csv` with your hand-labels, and your development and held-out scores side by side.
* Your test set (20 development + 8 held-out questions) and your Gem scoring sheet (Version 1).
* `retrieval_sweep.csv` and the comparison table.

## Reflection
1. Which failures were retrieval problems and which were generation problems? How did that change what you tried next?
2. Did your instruction file fix the failures you meant it to fix? Did any rule backfire (for example, refusing when the answer was there)?
3. How did the unanswerable questions behave in each version? What would a customer experience?
4. What did chunk size and K do to retrieval? Was your prediction right?
5. How did your final version do on the held-out questions compared with the development questions? If it did worse, what does that tell you about how you tuned it?
6. Given accuracy, build time and what you would have to maintain, which version would you recommend a firm deploy, and what would change your mind?

*Keep your assistant working: in Session 22 another team will audit it.*